# Class Decorators — Advanced Problems with Solutions
## Tutorial-Style Practice Set 2

This notebook is a **new** set of problems on class decorators.

The style is intentionally tutorial-like:

- we start with a small version,
- inspect what Python actually stored in the class,
- run it,
- discover limitations,
- improve the implementation,
- and only then arrive at the final solution.

So, instead of jumping straight to a polished answer, we will often ask:

> What happened?

and then use Python's introspection tools to find out.

The problems are advanced, but each one is broken into logical steps.

We will focus on a few recurring ideas.

A class decorator receives a class object:

```python
def decorator(cls):
    ...
    return cls
```

But "doing something to a class" can mean many different things.

We might:

- inject attributes,
- inject methods,
- replace methods,
- inspect raw descriptors,
- rebuild properties,
- wrap the class itself,
- enforce rules,
- collect metadata,
- or even return a replacement class.

The important thing is to understand **what object we are modifying** and **what Python has stored in the class namespace**.

In [1]:
from __future__ import annotations

from functools import wraps
import inspect
import copy
from collections import defaultdict
from typing import get_type_hints

print("Ready.")

Ready.


We'll use `vars(cls)` quite a lot.

Why?

Because `vars(cls)` gives us the class namespace **without triggering normal attribute binding**.

That matters for descriptors such as:

- `staticmethod`
- `classmethod`
- `property`

Let's quickly remind ourselves what those look like.

In [2]:
class WarmUp:
    def instance_method(self):
        return "instance"

    @staticmethod
    def static_method():
        return "static"

    @classmethod
    def class_method(cls):
        return cls.__name__

    @property
    def value(self):
        return 100

In [3]:
for name in ("instance_method", "static_method", "class_method", "value"):
    raw = vars(WarmUp)[name]
    print(f"{name:16} -> {type(raw).__name__}")

instance_method  -> function
static_method    -> staticmethod
class_method     -> classmethod
value            -> property


Notice that the raw class dictionary contains different kinds of objects.

That distinction will become important in several problems below.

---
# Problem 1 — A Declarative Plugin Registry

Suppose we are building a small application with multiple "handlers".

We want classes to register themselves simply by writing:

```python
@plugin("csv")
class CsvHandler:
    ...
```

The first version sounds easy enough.

Let's start with a global registry and a simple parameterized class decorator.

In [4]:
PLUGINS = {}

def plugin(name):
    def decorator(cls):
        PLUGINS[name] = cls
        return cls
    return decorator

Now let's define a few classes.

In [5]:
@plugin("csv")
class CsvHandler:
    def handle(self, data):
        return f"CSV:{data}"

@plugin("json")
class JsonHandler:
    def handle(self, data):
        return f"JSON:{data}"

In [6]:
PLUGINS

{'csv': __main__.CsvHandler, 'json': __main__.JsonHandler}

So far, so good.

But there is a problem.

What if two classes accidentally use the same registration name?

In [7]:
@plugin("csv")
class AnotherCsvHandler:
    def handle(self, data):
        return f"OTHER CSV:{data}"

In [8]:
PLUGINS["csv"]

__main__.AnotherCsvHandler

Oops.

The first class was silently replaced.

That may be acceptable in some systems, but for a plugin registry it is often a bug.

Let's make the decorator stricter.

We also want:

- a non-empty string key,
- duplicate detection,
- and a little bit of metadata on the class itself.

In [9]:
PLUGINS = {}

def plugin(name):
    if not isinstance(name, str) or not name.strip():
        raise ValueError("plugin name must be a non-empty string")

    key = name.strip()

    def decorator(cls):
        if key in PLUGINS:
            raise KeyError(f"plugin {key!r} is already registered")

        PLUGINS[key] = cls
        cls.plugin_name = key
        return cls

    return decorator

In [10]:
@plugin("csv")
class CsvHandler:
    def handle(self, data):
        return f"CSV:{data}"

@plugin("json")
class JsonHandler:
    def handle(self, data):
        return f"JSON:{data}"

handler_cls = PLUGINS["json"]
handler = handler_cls()

assert handler.handle("abc") == "JSON:abc"
assert JsonHandler.plugin_name == "json"

print(PLUGINS)

{'csv': <class '__main__.CsvHandler'>, 'json': <class '__main__.JsonHandler'>}


### Final observation

This is a nice example of a class decorator that performs a **side effect at class definition time**.

The class is still returned unchanged, but the decorator also updates external state.

This pattern appears in:

- serialization frameworks,
- command routers,
- plugin systems,
- test discovery,
- ORM model registration,
- web framework routing.

---
# Problem 2 — Automatically Build a Command Table

Let's make the previous idea more interesting.

Instead of registering whole classes, we want a class decorator to inspect methods marked as commands.

Usage should look like this:

```python
class Shell:
    @command("hello")
    def greet(...):
        ...
```

Then:

```python
@collect_commands
class Shell:
    ...
```

should create:

```python
Shell.__commands__
```

mapping command names to method names.

We need two decorators here:

1. a **function decorator** that only stores metadata,
2. a **class decorator** that scans the class namespace.

In [11]:
def command(name):
    def decorator(fn):
        fn.__command_name__ = name
        return fn
    return decorator

Notice that the function decorator does not wrap the function at all.

It just adds metadata.

Now let's write a simple collector.

In [12]:
def collect_commands(cls):
    commands = {}

    for name, obj in vars(cls).items():
        command_name = getattr(obj, "__command_name__", None)
        if command_name is not None:
            commands[command_name] = name

    cls.__commands__ = commands
    return cls

In [13]:
@collect_commands
class Shell:
    @command("hello")
    def greet(self, person):
        return f"Hello {person}"

    @command("bye")
    def farewell(self, person):
        return f"Bye {person}"

In [14]:
Shell.__commands__

{'hello': 'greet', 'bye': 'farewell'}

That works for ordinary instance methods.

But let's try a static method.

In [15]:
@collect_commands
class UtilityShell:
    @staticmethod
    @command("version")
    def version():
        return "1.0"

In [16]:
UtilityShell.__commands__

{}

Interesting — that one works.

Why?

Because `@command` was applied first, so the metadata lives on the underlying function, and `staticmethod` delegates some attribute access to it in modern Python implementations.

But relying on such details is not a great design.

Also, what happens with `classmethod`?

Let's inspect the raw objects directly.

In [17]:
class InspectCommands:
    @staticmethod
    @command("s")
    def s():
        pass

    @classmethod
    @command("c")
    def c(cls):
        pass

for name in ("s", "c"):
    raw = vars(InspectCommands)[name]
    print(name, type(raw).__name__, getattr(raw, "__command_name__", None))
    print("underlying:", getattr(raw.__func__, "__command_name__", None))

s staticmethod None
underlying: s
c classmethod None
underlying: c


The robust solution is obvious now:

- if the object is a `staticmethod` or `classmethod`,
- inspect `obj.__func__`.

Let's improve the class decorator.

In [18]:
def collect_commands(cls):
    commands = {}

    for name, obj in vars(cls).items():
        if isinstance(obj, (staticmethod, classmethod)):
            target = obj.__func__
        else:
            target = obj

        command_name = getattr(target, "__command_name__", None)

        if command_name is not None:
            if command_name in commands:
                raise ValueError(
                    f"duplicate command {command_name!r} in {cls.__name__}"
                )
            commands[command_name] = name

    cls.__commands__ = commands
    return cls

In [19]:
@collect_commands
class AdvancedShell:
    @command("hello")
    def greet(self, person):
        return f"Hello {person}"

    @staticmethod
    @command("version")
    def version():
        return "2.0"

    @classmethod
    @command("whoami")
    def whoami(cls):
        return cls.__name__

In [20]:
assert AdvancedShell.__commands__ == {
    "hello": "greet",
    "version": "version",
    "whoami": "whoami",
}

print(AdvancedShell.__commands__)

{'hello': 'greet', 'version': 'version', 'whoami': 'whoami'}


This gives us an important general pattern:

> Function decorators can mark methods, and a class decorator can later collect those markers.

This is one way to build declarative APIs without metaclasses.

---
# Problem 3 — Add a `dispatch` Method Automatically

We now have a command table.

But users still have to manually look up the method.

Let's make the class decorator inject a method:

```python
shell.dispatch("hello", "Ada")
```

The `dispatch` method should:

1. look up the command,
2. get the corresponding bound attribute from the instance,
3. call it,
4. raise a clear `KeyError` for unknown commands.

We already know how to inject methods: functions assigned to a class become descriptors and bind to instances.

Let's build on the previous decorator.

In [21]:
def command_dispatcher(cls):
    # First collect command metadata.
    collect_commands(cls)

    if "dispatch" in vars(cls):
        raise TypeError(f"{cls.__name__} already defines dispatch")

    def dispatch(self, command_name, *args, **kwargs):
        try:
            method_name = self.__commands__[command_name]
        except KeyError:
            available = ", ".join(sorted(self.__commands__))
            raise KeyError(
                f"unknown command {command_name!r}; available: {available}"
            ) from None

        target = getattr(self, method_name)
        return target(*args, **kwargs)

    cls.dispatch = dispatch
    return cls

In [22]:
@command_dispatcher
class CommandApp:
    @command("add")
    def add(self, a, b):
        return a + b

    @staticmethod
    @command("upper")
    def upper(value):
        return value.upper()

In [23]:
app = CommandApp()

assert app.dispatch("add", 10, 20) == 30
assert app.dispatch("upper", "hello") == "HELLO"

print(app.dispatch("add", 10, 20))
print(app.dispatch("upper", "hello"))

30
HELLO


Let's also check the failure case.

Good decorators should not only make the happy path work — they should produce useful errors.

In [24]:
try:
    app.dispatch("missing")
except KeyError as ex:
    print(ex)

"unknown command 'missing'; available: add, upper"


This problem combines three ideas:

- function metadata,
- class namespace inspection,
- method injection.

And notice that `getattr(self, method_name)` intentionally **does** invoke normal descriptor behavior.

Earlier we used `vars(cls)` because we wanted raw objects.

Here we want a bound method, so `getattr` is exactly the right tool.

---
# Problem 4 — A Selective Method Tracer

Now let's return to class-wide wrapping.

We want:

```python
@trace_public
class Service:
    ...
```

to trace every **public** method.

Let's first write a normal function decorator.

In [25]:
def trace(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        print(f"enter: {fn.__qualname__}")
        result = fn(*args, **kwargs)
        print(f"exit:  {fn.__qualname__} -> {result!r}")
        return result

    return wrapper

The naive class decorator might use `callable`.

Let's try it.

In [26]:
def naive_trace_public(cls):
    for name, obj in list(vars(cls).items()):
        if not name.startswith("_") and callable(obj):
            setattr(cls, name, trace(obj))
    return cls

In [27]:
class CallableObject:
    def __call__(self):
        return "called"

@naive_trace_public
class Example:
    helper = CallableObject()

    class Nested:
        pass

    def work(self):
        return "work"

In [28]:
print("helper:", Example.helper)
print("Nested:", Example.Nested)
print("work:", Example.work)

helper: <function trace.<locals>.wrapper at 0x000001D5402B5EE0>
Nested: <function Example.Nested at 0x000001D5402B5F80>
work: <function Example.work at 0x000001D5402B6160>


This is the classic `callable` problem.

Classes are callable.

Instances can also be callable.

So our decorator has replaced things we did not intend to treat as methods.

Let's restrict ourselves to actual function objects.

In [29]:
def trace_public(cls):
    for name, obj in list(vars(cls).items()):
        if name.startswith("_"):
            continue

        if inspect.isfunction(obj):
            setattr(cls, name, trace(obj))

    return cls

In [30]:
@trace_public
class SaferExample:
    helper = CallableObject()

    class Nested:
        pass

    def work(self):
        return "work"

x = SaferExample()

assert isinstance(SaferExample.helper, CallableObject)
assert inspect.isclass(SaferExample.Nested)
assert x.work() == "work"

enter: SaferExample.work
exit:  SaferExample.work -> 'work'


Much better.

But this still ignores static methods and class methods because their raw class attributes are descriptor objects rather than plain functions.

We'll fix that next.

---
# Problem 5 — Preserve `staticmethod` and `classmethod`

Let's start with a class containing all three method styles.

In [31]:
class MethodKinds:
    def instance(self, x):
        return x + 1

    @staticmethod
    def static(x):
        return x * 2

    @classmethod
    def class_name(cls):
        return cls.__name__

In [32]:
for name, obj in vars(MethodKinds).items():
    if name in {"instance", "static", "class_name"}:
        print(name, "->", type(obj).__name__)

instance -> function
static -> staticmethod
class_name -> classmethod


The raw namespace tells us exactly what we need to do.

For static and class methods:

1. keep the descriptor type,
2. extract `.__func__`,
3. decorate the function,
4. rebuild the descriptor.

In [33]:
def trace_all_method_kinds(cls):
    for name, obj in list(vars(cls).items()):
        if name.startswith("_"):
            continue

        if isinstance(obj, (staticmethod, classmethod)):
            descriptor_type = type(obj)
            decorated = trace(obj.__func__)
            setattr(cls, name, descriptor_type(decorated))

        elif inspect.isfunction(obj):
            setattr(cls, name, trace(obj))

    return cls

In [34]:
@trace_all_method_kinds
class TracedMethods:
    def instance(self, x):
        return x + 1

    @staticmethod
    def static(x):
        return x * 2

    @classmethod
    def class_name(cls):
        return cls.__name__

In [35]:
obj = TracedMethods()

assert obj.instance(10) == 11
assert TracedMethods.static(10) == 20
assert TracedMethods.class_name() == "TracedMethods"

assert isinstance(vars(TracedMethods)["static"], staticmethod)
assert isinstance(vars(TracedMethods)["class_name"], classmethod)

print("descriptor types preserved")

enter: TracedMethods.instance
exit:  TracedMethods.instance -> 11
enter: TracedMethods.static
exit:  TracedMethods.static -> 20
enter: TracedMethods.class_name
exit:  TracedMethods.class_name -> 'TracedMethods'
descriptor types preserved


This is one of the most important best practices for class decorators that transform methods:

> do not throw away descriptor semantics.

If a method started life as a `classmethod`, it should still be a `classmethod` after decoration.

---
# Problem 6 — Extend the Tracer to Properties

Properties are descriptors too, but they are different.

A `property` may contain up to three functions:

- `fget`
- `fset`
- `fdel`

Let's inspect one.

In [36]:
class Person:
    def __init__(self, name):
        self._name = name

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        self._name = value

    @name.deleter
    def name(self):
        del self._name

prop = vars(Person)["name"]

print(type(prop).__name__)
print(prop.fget)
print(prop.fset)
print(prop.fdel)

property
<function Person.name at 0x000001D5402B6980>
<function Person.name at 0x000001D5402B6B60>
<function Person.name at 0x000001D5402B6480>


Could we simply assign to `prop.fget`?

Let's see.

In [37]:
try:
    prop.fget = trace(prop.fget)
except Exception as ex:
    print(type(ex).__name__, ex)

AttributeError readonly attribute


So we need to build a new property.

We can do that directly using:

```python
property(fget, fset, fdel, doc)
```

Let's first write a helper that transforms a property.

In [38]:
def trace_property(prop):
    fget = trace(prop.fget) if prop.fget is not None else None
    fset = trace(prop.fset) if prop.fset is not None else None
    fdel = trace(prop.fdel) if prop.fdel is not None else None

    return property(fget, fset, fdel, prop.__doc__)

Now we can add one more branch to the class decorator.

In [39]:
def trace_public_members(cls):
    for name, obj in list(vars(cls).items()):
        if name.startswith("_"):
            continue

        if isinstance(obj, (staticmethod, classmethod)):
            descriptor_type = type(obj)
            setattr(cls, name, descriptor_type(trace(obj.__func__)))

        elif isinstance(obj, property):
            setattr(cls, name, trace_property(obj))

        elif inspect.isfunction(obj):
            setattr(cls, name, trace(obj))

    return cls

In [40]:
@trace_public_members
class Account:
    def __init__(self, balance):
        self._balance = balance

    def deposit(self, amount):
        self._balance += amount
        return self._balance

    @property
    def balance(self):
        '''Current account balance.'''
        return self._balance

    @balance.setter
    def balance(self, value):
        self._balance = value

    @staticmethod
    def bank():
        return "Example Bank"

    @classmethod
    def type_name(cls):
        return cls.__name__

In [41]:
a = Account(100)

assert a.deposit(25) == 125
assert a.balance == 125

a.balance = 200
assert a.balance == 200

assert Account.bank() == "Example Bank"
assert Account.type_name() == "Account"
assert vars(Account)["balance"].__doc__ == "Current account balance."

print("property metadata preserved")

enter: Account.deposit
exit:  Account.deposit -> 125
enter: Account.balance
exit:  Account.balance -> 125
enter: Account.balance
exit:  Account.balance -> None
enter: Account.balance
exit:  Account.balance -> 200
enter: Account.bank
exit:  Account.bank -> 'Example Bank'
enter: Account.type_name
exit:  Account.type_name -> 'Account'
property metadata preserved


At this point we have something much closer to a general member-transforming class decorator.

But we still need to think about repeated decoration.

---
# Problem 7 — Make Class-Wide Decoration Idempotent

Suppose a class decorator is applied twice.

Maybe this happens because:

- one decorator calls another,
- a framework processes a class twice,
- or a user accidentally repeats the decorator.

If we blindly wrap methods again, we get nested wrappers.

Let's demonstrate.

In [42]:
def marker_wrap(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        print("wrapped")
        return fn(*args, **kwargs)
    return wrapper

def wrap_methods(cls):
    for name, obj in list(vars(cls).items()):
        if inspect.isfunction(obj):
            setattr(cls, name, marker_wrap(obj))
    return cls

In [43]:
@wrap_methods
class RepeatDemo:
    def ping(self):
        return "pong"

first = vars(RepeatDemo)["ping"]
wrap_methods(RepeatDemo)
second = vars(RepeatDemo)["ping"]

print("same object?", first is second)

same object? False


They are different.

That means the second pass wrapped the already wrapped function.

A common solution is to attach a sentinel attribute to the wrapper.

In [44]:
def trace_once(fn):
    if getattr(fn, "__traced_once__", False):
        return fn

    @wraps(fn)
    def wrapper(*args, **kwargs):
        print(f"trace-once: {fn.__qualname__}")
        return fn(*args, **kwargs)

    wrapper.__traced_once__ = True
    return wrapper

Now let's write an idempotent class decorator.

We need to check the underlying functions for descriptors too.

In [45]:
def trace_class_once(cls):
    for name, obj in list(vars(cls).items()):
        if name.startswith("_"):
            continue

        if isinstance(obj, (staticmethod, classmethod)):
            descriptor_type = type(obj)
            transformed = trace_once(obj.__func__)
            setattr(cls, name, descriptor_type(transformed))

        elif isinstance(obj, property):
            fget = trace_once(obj.fget) if obj.fget else None
            fset = trace_once(obj.fset) if obj.fset else None
            fdel = trace_once(obj.fdel) if obj.fdel else None
            setattr(cls, name, property(fget, fset, fdel, obj.__doc__))

        elif inspect.isfunction(obj):
            setattr(cls, name, trace_once(obj))

    return cls

In [46]:
@trace_class_once
class IdempotentDemo:
    def ping(self):
        return "pong"

before = vars(IdempotentDemo)["ping"]
trace_class_once(IdempotentDemo)
after = vars(IdempotentDemo)["ping"]

assert before is after
assert IdempotentDemo().ping() == "pong"

print("same object after second pass:", before is after)

trace-once: IdempotentDemo.ping
same object after second pass: True


Idempotence is not always necessary.

But when a decorator may be used by a framework or reused compositionally, it is a very useful property.

---
# Problem 8 — Enforce Constructor Type Annotations

Let's switch to a different kind of class decorator.

Suppose a class uses annotations:

```python
class User:
    name: str
    age: int
```

We want a decorator to wrap `__init__` and validate the resulting instance.

This gives us a nice opportunity to combine:

- annotations,
- class inspection,
- method wrapping,
- validation errors.

Let's begin with a very small version.

After `__init__` finishes, inspect annotated attributes on the instance.

In [47]:
def validate_annotations(cls):
    annotations = get_type_hints(cls)
    original_init = cls.__init__

    @wraps(original_init)
    def wrapped_init(self, *args, **kwargs):
        original_init(self, *args, **kwargs)

        for name, expected_type in annotations.items():
            if not hasattr(self, name):
                raise AttributeError(
                    f"{cls.__name__}.__init__ did not set annotated attribute {name!r}"
                )

            value = getattr(self, name)

            if not isinstance(value, expected_type):
                raise TypeError(
                    f"{name} must be {expected_type.__name__}, "
                    f"got {type(value).__name__}"
                )

    cls.__init__ = wrapped_init
    return cls

In [48]:
@validate_annotations
class User:
    name: str
    age: int

    def __init__(self, name, age):
        self.name = name
        self.age = age

In [49]:
u = User("Ada", 36)

assert u.name == "Ada"
assert u.age == 36

try:
    User("Ada", "thirty-six")
except TypeError as ex:
    print(ex)

age must be int, got str


This works for simple runtime-checkable types.

But we should be precise about the scope.

`typing` constructs such as `list[int]`, unions, protocols, and other advanced annotations may require more sophisticated logic.

For this exercise, we deliberately support only annotation objects accepted by `isinstance`.

Let's make the failure clearer if unsupported annotations are used.

In [50]:
def validate_simple_annotations(cls):
    annotations = get_type_hints(cls)

    for name, expected_type in annotations.items():
        try:
            isinstance(None, expected_type)
        except TypeError:
            raise TypeError(
                f"annotation for {name!r} is not a simple runtime-checkable type"
            ) from None

    original_init = cls.__init__

    @wraps(original_init)
    def wrapped_init(self, *args, **kwargs):
        original_init(self, *args, **kwargs)

        for name, expected_type in annotations.items():
            if not hasattr(self, name):
                raise AttributeError(
                    f"missing annotated attribute {name!r}"
                )

            value = getattr(self, name)

            if not isinstance(value, expected_type):
                raise TypeError(
                    f"{name} expected {expected_type.__name__}; "
                    f"received {type(value).__name__}"
                )

    cls.__init__ = wrapped_init
    return cls

In [51]:
@validate_simple_annotations
class Product:
    sku: str
    price: float

    def __init__(self, sku, price):
        self.sku = sku
        self.price = price

p = Product("P-100", 19.95)
assert p.price == 19.95

print("validated:", p.sku, p.price)

validated: P-100 19.95


This decorator modifies behavior without changing the public constructor signature metadata because `functools.wraps` preserves useful attributes from the original `__init__`.

---
# Problem 9 — Freeze Instances After Initialization

Let's make a class decorator that allows normal assignment during `__init__`, but prevents adding new attributes afterward.

Existing attributes may still be modified.

Desired behavior:

```python
p = Point(1, 2)
p.x = 10      # allowed
p.z = 99      # error
```

This requires us to coordinate `__init__` and `__setattr__`.

Let's reason through the states.

During construction we need:

```text
_frozen = False
```

After construction:

```text
_frozen = True
```

Then `__setattr__` can check whether the attribute already exists.

In [52]:
def freeze_shape(cls):
    original_init = cls.__init__
    original_setattr = cls.__setattr__

    @wraps(original_init)
    def wrapped_init(self, *args, **kwargs):
        original_setattr(self, "_shape_frozen", False)
        original_init(self, *args, **kwargs)
        original_setattr(self, "_shape_frozen", True)

    def wrapped_setattr(self, name, value):
        frozen = getattr(self, "_shape_frozen", False)

        if frozen and not hasattr(self, name):
            raise AttributeError(
                f"{type(self).__name__} is frozen; "
                f"cannot add attribute {name!r}"
            )

        return original_setattr(self, name, value)

    cls.__init__ = wrapped_init
    cls.__setattr__ = wrapped_setattr
    return cls

In [53]:
@freeze_shape
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

In [54]:
p = Point(1, 2)

p.x = 100
assert p.x == 100

try:
    p.z = 300
except AttributeError as ex:
    print(ex)

Point is frozen; cannot add attribute 'z'


There is an interesting design decision here.

We are not making the object immutable.

We are only freezing its **shape**.

That is, the set of attribute names becomes fixed after initialization.

This can be useful for catching typos:

```python
obj.totl = 10
```

instead of:

```python
obj.total = 10
```

### A subtle point

Our decorator stores `_shape_frozen` on the instance itself.

Because the flag exists before the object is frozen, later updates to that existing attribute are technically allowed.

In production code we might hide this state more carefully, but for this exercise it makes the mechanism easy to inspect and understand.

---
# Problem 10 — Transactional Methods with Rollback

Now let's build something more advanced.

Suppose an object has mutable state in `self.__dict__`.

We want to mark selected methods as transactional:

```python
@transactional
def transfer(...):
    ...
```

If the method raises an exception, we want the instance state restored to what it was before the call.

We will use two stages again:

1. a method marker,
2. a class decorator that finds marked methods and wraps them.

In [55]:
def transactional(fn):
    fn.__transactional__ = True
    return fn

Now let's write the wrapper.

Before the method runs, take a deep copy of `self.__dict__`.

If the method fails:

1. clear the current state,
2. restore the snapshot,
3. re-raise the original exception.

In [56]:
def transactional_class(cls):
    for name, obj in list(vars(cls).items()):
        if not inspect.isfunction(obj):
            continue

        if not getattr(obj, "__transactional__", False):
            continue

        @wraps(obj)
        def wrapper(self, *args, __fn=obj, **kwargs):
            snapshot = copy.deepcopy(self.__dict__)

            try:
                return __fn(self, *args, **kwargs)
            except Exception:
                self.__dict__.clear()
                self.__dict__.update(snapshot)
                raise

        setattr(cls, name, wrapper)

    return cls

Notice the slightly unusual default argument:

```python
__fn=obj
```

Why did we do that?

Because the wrapper is created inside a loop.

Without binding the current function immediately, every wrapper could accidentally refer to the final `obj` from the loop.

This is the classic late-binding closure issue.

In [57]:
@transactional_class
class Wallet:
    def __init__(self, balance):
        self.balance = balance
        self.history = []

    @transactional
    def withdraw(self, amount):
        self.balance -= amount
        self.history.append(("withdraw", amount))

        if self.balance < 0:
            raise ValueError("insufficient funds")

        return self.balance

    def deposit(self, amount):
        self.balance += amount
        self.history.append(("deposit", amount))
        return self.balance

In [58]:
w = Wallet(100)

assert w.withdraw(40) == 60
assert w.balance == 60
assert w.history == [("withdraw", 40)]

before = copy.deepcopy(w.__dict__)

try:
    w.withdraw(1000)
except ValueError as ex:
    print(ex)

assert w.__dict__ == before
print("state restored:", w.__dict__)

insufficient funds
state restored: {'balance': 60, 'history': [('withdraw', 40)]}


This is deliberately a simplified transaction system.

It only knows about state reachable through `self.__dict__`, and `deepcopy` itself has limitations.

But it is a good example of a class decorator coordinating metadata and method transformation.

---
# Problem 11 — Per-Method Call Statistics

Suppose we want every public method to maintain a call count.

We want the statistics stored on the class:

```python
Service.__call_counts__
```

Let's build it step by step.

First, the class decorator creates one dictionary.

In [59]:
def count_calls(cls):
    counts = defaultdict(int)
    cls.__call_counts__ = counts

    return cls

Now we need wrappers that update that shared dictionary.

Again, be careful with loop variables.

In [60]:
def count_calls(cls):
    counts = defaultdict(int)
    cls.__call_counts__ = counts

    def make_wrapper(fn, member_name):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            counts[member_name] += 1
            return fn(*args, **kwargs)

        return wrapper

    for name, obj in list(vars(cls).items()):
        if name.startswith("_"):
            continue

        if inspect.isfunction(obj):
            setattr(cls, name, make_wrapper(obj, name))

    return cls

In [61]:
@count_calls
class BasicService:
    def a(self):
        return "a"

    def b(self):
        return "b"

In [62]:
s = BasicService()

s.a()
s.a()
s.b()

print(dict(BasicService.__call_counts__))

{'a': 2, 'b': 1}


Now let's add static methods and class methods.

The same descriptor-preservation rule applies.

In [63]:
def count_all_calls(cls):
    counts = defaultdict(int)
    cls.__call_counts__ = counts

    def make_wrapper(fn, member_name):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            counts[member_name] += 1
            return fn(*args, **kwargs)
        return wrapper

    for name, obj in list(vars(cls).items()):
        if name.startswith("_"):
            continue

        if isinstance(obj, (staticmethod, classmethod)):
            descriptor_type = type(obj)
            wrapped = make_wrapper(obj.__func__, name)
            setattr(cls, name, descriptor_type(wrapped))

        elif inspect.isfunction(obj):
            setattr(cls, name, make_wrapper(obj, name))

    return cls

In [64]:
@count_all_calls
class Service:
    def ping(self):
        return "pong"

    @staticmethod
    def add(a, b):
        return a + b

    @classmethod
    def service_name(cls):
        return cls.__name__

In [65]:
service = Service()

service.ping()
service.ping()

Service.add(1, 2)

Service.service_name()
Service.service_name()
Service.service_name()

assert dict(Service.__call_counts__) == {
    "ping": 2,
    "add": 1,
    "service_name": 3,
}

print(dict(Service.__call_counts__))

{'ping': 2, 'add': 1, 'service_name': 3}


This pattern is useful for lightweight instrumentation.

A production metrics system would probably send data somewhere else instead of storing it directly on the class, but the transformation principle is the same.

---
# Problem 12 — Class Replacement: One Instance Per Key

So far almost every decorator has returned the same class object.

But a class decorator may return **any object**.

In particular, it can return a new class.

Let's use that to build a "one instance per key" cache.

For example:

```python
@instance_per_key(lambda name: name.lower())
class User:
    ...
```

Then two constructions with the same logical key should return the same instance.

This is more invasive than the previous decorators.

We are going to return a subclass that controls `__new__`.

Let's first implement the caching part.

In [66]:
def instance_per_key(key_func):
    def decorator(cls):
        cache = {}

        class Cached(cls):
            def __new__(subcls, *args, **kwargs):
                key = key_func(*args, **kwargs)

                if key in cache:
                    return cache[key]

                obj = super().__new__(subcls)
                cache[key] = obj
                return obj

        Cached.__name__ = cls.__name__
        Cached.__qualname__ = cls.__qualname__
        Cached.__module__ = cls.__module__
        Cached.__doc__ = cls.__doc__
        Cached.__wrapped__ = cls
        Cached.__instance_cache__ = cache

        return Cached

    return decorator

Let's test it.

In [67]:
@instance_per_key(lambda name: name.lower())
class CachedUser:
    def __init__(self, name):
        self.name = name

In [68]:
u1 = CachedUser("Ada")
u2 = CachedUser("ADA")

print(u1 is u2)
print(u1.name, u2.name)

True
ADA ADA


We got the same object.

But something slightly surprising happened.

`__init__` ran again on the cached instance, so the second construction modified its state.

That is often undesirable.

We need to prevent repeated initialization for the same key.

In [69]:
def instance_per_key(key_func):
    def decorator(cls):
        cache = {}
        initialized = set()

        class Cached(cls):
            def __new__(subcls, *args, **kwargs):
                key = key_func(*args, **kwargs)

                if key in cache:
                    return cache[key]

                obj = super().__new__(subcls)
                obj.__cache_key__ = key
                cache[key] = obj
                return obj

            def __init__(self, *args, **kwargs):
                key = self.__cache_key__

                if key in initialized:
                    return

                super().__init__(*args, **kwargs)
                initialized.add(key)

        Cached.__name__ = cls.__name__
        Cached.__qualname__ = cls.__qualname__
        Cached.__module__ = cls.__module__
        Cached.__doc__ = cls.__doc__
        Cached.__wrapped__ = cls
        Cached.__instance_cache__ = cache

        return Cached

    return decorator

In [70]:
@instance_per_key(lambda name: name.lower())
class UserByName:
    def __init__(self, name):
        self.name = name
        self.init_count = getattr(self, "init_count", 0) + 1

In [71]:
u1 = UserByName("Ada")
u2 = UserByName("ADA")
u3 = UserByName("Grace")

assert u1 is u2
assert u1 is not u3
assert u1.init_count == 1
assert u1.name == "Ada"

print("same key:", u1 is u2)
print("different key:", u1 is not u3)
print("initializations:", u1.init_count)

same key: True
different key: True
initializations: 1


### Why this problem matters

Class replacement is powerful, but it changes the design space significantly.

Once a decorator returns a replacement class, you should think carefully about:

- type identity,
- inheritance,
- pickling,
- introspection,
- construction semantics,
- repeated initialization,
- and whether users expect the original class object.

Mutation is often simpler.

Replacement should be intentional.

---
# Problem 13 — Validate a Class Contract at Definition Time

Let's build a decorator that rejects classes that do not provide required methods.

Usage:

```python
@requires("start", "stop")
class Worker:
    ...
```

The error should happen immediately when the class is defined.

This is useful when class creation itself should enforce a protocol-like rule.

In [72]:
def requires(*names):
    def decorator(cls):
        missing = []

        for name in names:
            if not hasattr(cls, name):
                missing.append(name)

        if missing:
            raise TypeError(
                f"{cls.__name__} is missing required members: {missing}"
            )

        return cls

    return decorator

That checks existence.

But maybe an attribute exists and is not callable.

In [73]:
try:
    @requires("start")
    class Broken:
        start = 100
except TypeError as ex:
    print(ex)
else:
    print("The naive decorator accepted it.")

The naive decorator accepted it.


So let's make the contract more precise.

We want required **methods**.

We can use `inspect.getattr_static` to retrieve attributes without invoking descriptors.

In [74]:
def requires_methods(*names):
    def decorator(cls):
        problems = []

        for name in names:
            try:
                raw = inspect.getattr_static(cls, name)
            except AttributeError:
                problems.append(f"{name}: missing")
                continue

            if isinstance(raw, (staticmethod, classmethod)):
                target = raw.__func__
            else:
                target = raw

            if not callable(target):
                problems.append(f"{name}: not callable")

        if problems:
            joined = "; ".join(problems)
            raise TypeError(f"{cls.__name__} violates contract: {joined}")

        return cls

    return decorator

In [75]:
@requires_methods("start", "stop")
class Worker:
    def start(self):
        return "started"

    @staticmethod
    def stop():
        return "stopped"

w = Worker()

assert w.start() == "started"
assert Worker.stop() == "stopped"

print("contract satisfied")

contract satisfied


This decorator does not modify the class at all.

It uses decoration purely as a **validation hook at definition time**.

That is another important use case for class decorators.

---
# Final Problem — Build a Small Auditing Framework

Let's put several ideas together.

We want:

```python
@audited
class Service:
    ...
```

with these rules:

1. public instance methods are audited,
2. public static methods are audited,
3. public class methods are audited,
4. public properties are audited,
5. arbitrary callable objects are ignored,
6. nested classes are ignored,
7. methods can opt out with `@audit_ignore`,
8. decoration is idempotent,
9. per-member call counts are stored on the class,
10. exceptions are logged and re-raised.

We'll build this gradually rather than writing the final version immediately.

First, let's create the opt-out marker.

Again, the marker itself does not wrap anything.

In [76]:
def audit_ignore(fn):
    fn.__audit_ignore__ = True
    return fn

Now let's write the function wrapper.

It needs access to:

- the class statistics dictionary,
- the member name.

So we'll use a small factory.

In [77]:
def make_audit_wrapper(fn, member_name, stats):
    if getattr(fn, "__audit_ignore__", False):
        return fn

    if getattr(fn, "__audited__", False):
        return fn

    @wraps(fn)
    def wrapper(*args, **kwargs):
        stats[member_name] += 1

        try:
            result = fn(*args, **kwargs)
        except Exception as ex:
            print(
                f"AUDIT {fn.__qualname__}: "
                f"raised {type(ex).__name__}: {ex}"
            )
            raise
        else:
            print(
                f"AUDIT {fn.__qualname__}: "
                f"returned {result!r}"
            )
            return result

    wrapper.__audited__ = True
    return wrapper

Now we need a helper for properties.

Each accessor should use the same member name so all property operations contribute to one counter.

In [78]:
def audit_property(prop, member_name, stats):
    fget = (
        make_audit_wrapper(prop.fget, member_name, stats)
        if prop.fget else None
    )
    fset = (
        make_audit_wrapper(prop.fset, member_name, stats)
        if prop.fset else None
    )
    fdel = (
        make_audit_wrapper(prop.fdel, member_name, stats)
        if prop.fdel else None
    )

    return property(fget, fset, fdel, prop.__doc__)

Now we can write the class decorator.

Notice the order of checks:

1. skip private names,
2. static/class method descriptors,
3. properties,
4. plain functions.

We do **not** use `callable`.

In [79]:
def audited(cls):
    stats = getattr(cls, "__audit_counts__", None)

    if stats is None:
        stats = defaultdict(int)
        cls.__audit_counts__ = stats

    for name, obj in list(vars(cls).items()):
        if name.startswith("_"):
            continue

        if isinstance(obj, (staticmethod, classmethod)):
            descriptor_type = type(obj)
            wrapped = make_audit_wrapper(
                obj.__func__,
                name,
                stats,
            )
            setattr(cls, name, descriptor_type(wrapped))

        elif isinstance(obj, property):
            setattr(
                cls,
                name,
                audit_property(obj, name, stats),
            )

        elif inspect.isfunction(obj):
            setattr(
                cls,
                name,
                make_audit_wrapper(obj, name, stats),
            )

    return cls

Let's create a class with many different member types.

This is where a class-wide decorator either proves itself or breaks.

In [80]:
@audited
class PaymentService:
    helper = CallableObject()

    class InternalType:
        pass

    def __init__(self, balance):
        self._balance = balance

    def charge(self, amount):
        if amount < 0:
            raise ValueError("amount cannot be negative")
        self._balance -= amount
        return self._balance

    @audit_ignore
    def health(self):
        return "ok"

    @property
    def balance(self):
        '''Current balance.'''
        return self._balance

    @balance.setter
    def balance(self, value):
        self._balance = value

    @staticmethod
    def currency():
        return "USD"

    @classmethod
    def service_name(cls):
        return cls.__name__

    def _private_helper(self):
        return "private"

Before running methods, let's inspect what survived the transformation.

In [81]:
print("helper type:", type(PaymentService.helper).__name__)
print("nested class:", inspect.isclass(PaymentService.InternalType))
print("currency raw type:", type(vars(PaymentService)["currency"]).__name__)
print("service_name raw type:", type(vars(PaymentService)["service_name"]).__name__)
print("balance raw type:", type(vars(PaymentService)["balance"]).__name__)

helper type: CallableObject
nested class: True
currency raw type: staticmethod
service_name raw type: classmethod
balance raw type: property


Great.

Now let's exercise the different branches.

In [82]:
service = PaymentService(100)

assert service.charge(25) == 75

assert service.balance == 75
service.balance = 200
assert service.balance == 200

assert PaymentService.currency() == "USD"
assert PaymentService.service_name() == "PaymentService"

assert service.health() == "ok"
assert service._private_helper() == "private"

AUDIT PaymentService.charge: returned 75
AUDIT PaymentService.balance: returned 75
AUDIT PaymentService.balance: returned None
AUDIT PaymentService.balance: returned 200
AUDIT PaymentService.currency: returned 'USD'
AUDIT PaymentService.service_name: returned 'PaymentService'


Let's deliberately trigger an exception too.

In [83]:
try:
    service.charge(-10)
except ValueError as ex:
    print("caught:", ex)

AUDIT PaymentService.charge: raised ValueError: amount cannot be negative
caught: amount cannot be negative


Now inspect the counts.

Notice that the failed `charge` still counts as a call.

Also, getter and setter operations on `balance` share the same key.

In [84]:
print(dict(PaymentService.__audit_counts__))

{'charge': 2, 'balance': 3, 'currency': 1, 'service_name': 1}


Let's make a few correctness assertions.

In [85]:
counts = dict(PaymentService.__audit_counts__)

assert counts["charge"] == 2
assert counts["balance"] == 3
assert counts["currency"] == 1
assert counts["service_name"] == 1

assert "health" not in counts
assert "_private_helper" not in counts

assert isinstance(PaymentService.helper, CallableObject)
assert inspect.isclass(PaymentService.InternalType)

assert isinstance(vars(PaymentService)["currency"], staticmethod)
assert isinstance(vars(PaymentService)["service_name"], classmethod)
assert isinstance(vars(PaymentService)["balance"], property)

print("all integrated checks passed")

all integrated checks passed


One final test: idempotence.

Calling `audited(PaymentService)` again should not replace already-audited functions with new wrappers.

In [86]:
before = vars(PaymentService)["charge"]

audited(PaymentService)

after = vars(PaymentService)["charge"]

assert before is after

print("idempotent:", before is after)

idempotent: True


That completes the integrated solution.

The final decorator now demonstrates most of the recurring best practices for class-wide method transformation:

- inspect the raw class dictionary,
- distinguish descriptors explicitly,
- unwrap and re-wrap static/class methods,
- rebuild properties,
- avoid broad `callable` checks,
- preserve metadata with `wraps`,
- support opt-out markers,
- bind loop variables safely,
- make repeated passes harmless,
- and test behavior rather than only inspecting syntax.

---
# Extra Practice Problems

The following smaller problems are included to give you more lines of code to experiment with.

Each one includes a compact solution, but try it yourself first.

## Extra 1 — Add a Class-Level `describe` Method

Create a decorator that injects:

```python
Class.describe()
```

returning the class name and all non-dunder names defined directly in the class.

In [87]:
def add_describe(cls):
    @classmethod
    def describe(inner_cls):
        names = sorted(
            name
            for name in vars(inner_cls)
            if not (name.startswith("__") and name.endswith("__"))
        )
        return {
            "class": inner_cls.__name__,
            "members": names,
        }

    cls.describe = describe
    return cls


@add_describe
class Sample:
    x = 10

    def work(self):
        return 20


info = Sample.describe()
print(info)

assert info["class"] == "Sample"
assert "x" in info["members"]
assert "work" in info["members"]

{'class': 'Sample', 'members': ['describe', 'work', 'x']}


## Extra 2 — Add a Creation Counter

Create a class decorator that counts how many instances were constructed.

The count should live on the class as:

```python
Class.instances_created
```

In [88]:
def count_instances(cls):
    cls.instances_created = 0
    original_init = cls.__init__

    @wraps(original_init)
    def wrapped_init(self, *args, **kwargs):
        original_init(self, *args, **kwargs)
        type(self).instances_created += 1

    cls.__init__ = wrapped_init
    return cls


@count_instances
class Ticket:
    def __init__(self, number):
        self.number = number


Ticket(1)
Ticket(2)
Ticket(3)

assert Ticket.instances_created == 3
print(Ticket.instances_created)

3


## Extra 3 — Reject Empty Classes

Write a class decorator that raises `TypeError` if the class defines no meaningful members of its own.

Ignore normal implementation attributes such as:

- `__module__`
- `__dict__`
- `__weakref__`
- `__doc__`

In [89]:
def reject_empty(cls):
    ignored = {
        "__module__",
        "__dict__",
        "__weakref__",
        "__doc__",
    }

    meaningful = [
        name
        for name in vars(cls)
        if name not in ignored
    ]

    if not meaningful:
        raise TypeError(f"{cls.__name__} is empty")

    return cls


@reject_empty
class NotEmpty:
    x = 1


try:
    @reject_empty
    class Empty:
        pass
except TypeError as ex:
    print(ex)

## Extra 4 — Rename a Class for Display

Write a decorator factory that adds a separate `display_name` without modifying `__name__`.

Why keep the two separate?

Because changing `__name__` can make introspection and debugging confusing.

In [90]:
def display_name(name):
    def decorator(cls):
        cls.display_name = name
        return cls
    return decorator


@display_name("Premium Customer")
class CustomerRecord:
    pass


assert CustomerRecord.__name__ == "CustomerRecord"
assert CustomerRecord.display_name == "Premium Customer"

print(CustomerRecord.__name__)
print(CustomerRecord.display_name)

CustomerRecord
Premium Customer


## Extra 5 — Count Only Successful Calls

Modify the counting pattern so a call is counted **only if the method returns successfully**.

In [91]:
def count_successes(cls):
    counts = defaultdict(int)
    cls.__success_counts__ = counts

    def wrap(fn, name):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            result = fn(*args, **kwargs)
            counts[name] += 1
            return result
        return wrapper

    for name, obj in list(vars(cls).items()):
        if inspect.isfunction(obj) and not name.startswith("_"):
            setattr(cls, name, wrap(obj, name))

    return cls


@count_successes
class Parser:
    def parse_int(self, value):
        return int(value)


parser = Parser()

assert parser.parse_int("10") == 10

try:
    parser.parse_int("not-an-int")
except ValueError:
    pass

assert Parser.__success_counts__["parse_int"] == 1
print(dict(Parser.__success_counts__))

{'parse_int': 1}


---
# What You Should Be Comfortable With Now

After working through this notebook, you should be able to reason about class decorators at several levels.

### Basic class mutation

You can:

- inject attributes,
- inject instance methods,
- inject class methods,
- attach metadata,
- validate classes when they are defined.

### Namespace inspection

You understand the difference between:

```python
vars(cls)[name]
```

and:

```python
getattr(cls, name)
```

and why raw descriptor inspection matters.

### Descriptor preservation

You can correctly transform:

- normal methods,
- `staticmethod`,
- `classmethod`,
- `property`.

### Safer transformation

You know why:

```python
callable(obj)
```

is often too broad when scanning a class namespace.

You can instead use explicit type checks and `inspect`.

### More advanced designs

You have also seen class decorators used for:

- command discovery,
- dispatching,
- validation,
- freezing instance shape,
- rollback behavior,
- call statistics,
- idempotent instrumentation,
- and even class replacement.

## Final best-practice checklist

When you write a class decorator, ask yourself:

1. Am I mutating the original class or replacing it?
2. Do I need raw namespace objects or bound attributes?
3. Could I accidentally touch nested classes or callable objects?
4. Am I preserving `staticmethod` / `classmethod` semantics?
5. Am I rebuilding properties correctly?
6. Am I preserving function metadata with `wraps`?
7. Could repeated decoration double-wrap my methods?
8. Are loop-created closures binding the correct function?
9. What happens when the wrapped method raises?
10. Have I tested edge cases, not just the happy path?

Those questions catch a large percentage of class-decorator bugs.